# Pattern 10: Agentic RAG

Follows this repo's mandatory 8-section notebook template -- this is one of "the 10 patterns."

**This notebook's committed execution uses `RAG_RECIPES_LLM=mock`** for both the embedding and
generation/agent-decision steps.

Every tool-call trace is logged to `outputs/agentic_traces.jsonl` so readers can see the agent's
reasoning -- section 5 below passes `trace_output_path` straight into `run_pattern()` (see
`evals/run.py`), so this happens as a natural side effect of the existing eval step, not a
separate/duplicate retrieval pass. Section 7 is a PENDING placeholder awaiting a
real-embeddings-and-LLM run.


## Reproducibility header

In [1]:
import platform
import subprocess
import sys

import numpy
import openai

print(f"platform: {platform.platform()}")
print(f"python: {sys.version}")
print(f"openai sdk: {openai.__version__}")
print(f"numpy: {numpy.__version__}")

try:
    git_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd="..").decode().strip()
except Exception:
    git_sha = "(not in a git repo checkout)"
print(f"git commit: {git_sha}")


platform: Windows-11-10.0.26200-SP0
python: 3.12.13 (main, Aug  7 2026, 02:26:41) [MSC v.1944 64 bit (AMD64)]
openai sdk: 2.53.0
numpy: 2.5.2
git commit: d8337d8fafa765c37169264abfbf530e3b1643e6


## Setup (loaded once, used by every section below)

In [2]:
import os

os.environ.setdefault("RAG_RECIPES_LLM", "mock")

from evals.run import load_corpus_by_id, load_qa_set, run_pattern
from recipes.llm import MockLLM, get_llm

corpus_by_id = load_corpus_by_id("../corpus/corpus.jsonl")
qa_set = load_qa_set("../evals/qa_set.jsonl")
llm = get_llm()  # used for generation (recipe_fn's own LLM calls)

# Judging needs its own backend: under a real API key this is the same
# real model, but under mock, `llm`'s canned generation text isn't valid
# JSON, and the judge prompts require JSON output. A separate MockLLM
# here demonstrates a clean, illustrative run instead of every question
# correctly (but noisily) failing to parse -- see evals/judges.py's
# JudgeParseError and evals/run.py's per-question error isolation.
if os.environ.get("RAG_RECIPES_LLM", "openai").lower() == "mock":
    judge_llm = MockLLM(default_response='{"score": 1, "reasoning": "Mock judge: looks fine."}')
else:
    judge_llm = llm

from recipes.embeddings import get_embedder

embedder = get_embedder()
trace_output_path = "../outputs/agentic_traces.jsonl"


## 1. What this pattern does

An LLM-driven loop (`prompts/agentic_prompt.txt`) decides, turn by turn, which tool to call --
`search_dense`, `search_bm25`, or `finish` -- rather than following a fixed retrieval strategy. The
agent's job is retrieval orchestration ONLY: `finish` ends the search loop but does not produce the
answer itself, so the final answer always goes through the exact same R4 held-constant generation
prompt every other pattern uses (see `recipes/agentic.py`'s module docstring for why -- keeping R4
airtight with zero exceptions was a deliberate design decision, not an oversight). Every step
(thought, action, action_input, observation) is recorded into `tool_call_trace`.


## 2. When to use it

- You don't know in advance what retrieval strategy (keyword vs. semantic, how many searches) a
  given question needs -- letting the model decide adaptively can outperform any single fixed
  pattern across a diverse question mix
- You want visibility into the retrieval reasoning itself (the trace), not just the final answer
- You can afford a variable, potentially larger number of LLM calls per question (bounded by
  `MAX_ITERATIONS`, default 4)


## 3. When NOT to use it

- This pattern's own key claim is blunt: "Most flexible. Hardest to debug." A malformed or
  looping agent is genuinely hard to diagnose compared to any fixed-strategy pattern -- `recipes/agentic.py`'s
  `_parse_agent_action()` defensively forces a `finish` on any unparseable response specifically to
  bound this risk, but the underlying unpredictability is real
- Cost/latency budget requires a predictable, fixed number of retrieval calls per question
- Your question mix is uniform enough that a fixed pattern (e.g. hybrid+rerank) already performs
  well -- the agent's adaptivity is wasted when there's nothing to adapt to


## 4. Implementation

In [3]:
from recipes.agentic import make_retrieve_and_answer

retrieve_and_answer = make_retrieve_and_answer(corpus_by_id, embedder=embedder, llm=llm)

# Try it on one question directly.
sample = retrieve_and_answer("What does PCEval stand for?", k=3)
print("retrieved:", sample.retrieved_chunk_ids)
print("answer:", sample.answer)
print("tool_call_trace steps:", len(sample.tool_call_trace))


retrieved: ['arxiv:2601.02404#0', 'arxiv:2601.02404#1', 'arxiv:2601.02404#2']
answer: PCEVAL stands for "Physical Computing Evaluation," which is a benchmark designed for evaluating the physical computing capabilities of large language models (LLMs) in both logical and physical aspects of projects involving hardware and software interaction [arxiv:2601.02404#0].
tool_call_trace steps: 4


## 5. Run on our eval set

In [4]:
pattern_fn = make_retrieve_and_answer(corpus_by_id, embedder=embedder, llm=llm)

result = run_pattern(
    recipe_fn=pattern_fn,
    qa_set=qa_set,
    corpus_by_id=corpus_by_id,
    llm=judge_llm,
    pattern_name="10_agentic",
    judges_enabled=True,
    trace_output_path=trace_output_path,
)

import os

print()
print(f"wrote traces to {trace_output_path}: {os.path.exists(trace_output_path)}")
with open(trace_output_path, encoding="utf-8") as f:
    n_trace_lines = sum(1 for _ in f)
print(f"trace lines written: {n_trace_lines}")


=== 10_agentic (n=18) ===
  hit@3: 1.000  [95% CI 1.000, 1.000]
  hit@10: 1.000  [95% CI 1.000, 1.000]
  mrr: 0.963  [95% CI 0.889, 1.000]
  faithfulness: 0.722  [95% CI 0.500, 0.944]
  answer_relevance: 0.944  [95% CI 0.833, 1.000]
  citation_accuracy: 0.769  [95% CI 0.657, 0.870]
  filter_accuracy: 0.000  [95% CI 0.000, 0.000]
  p50_latency_ms: 11085.1
  p95_latency_ms: 17390.7
  usd_per_query: $0.00867
  eval_usd: $0.1560

wrote traces to ../outputs/agentic_traces.jsonl: True
trace lines written: 18


## 6. Example query walkthrough

One example per eval-set category, showing the retrieved chunks, the (mocked) final answer, AND the
full agent trace -- seeing the agent's reasoning is the whole point of this pattern.

In [5]:
examples = {
    "keyword": "What does PCEval stand for?",
    "paraphrase": "Why do repeated image generations from the same text prompt in diffusion models end up looking so similar to each other?",
    "multi_hop": "The two photonics-AI-systems papers in this corpus each address a different part of the same challenge. What does each one focus on?",
    "filter": "Among the cs.LG papers in this corpus, which one addresses diagnosing a pregnancy complication using deep learning?",
}

for category, question in examples.items():
    result = retrieve_and_answer(question, k=3)
    print(f"--- {category} ---")
    print(f"Q: {question}")
    print(f"Retrieved: {result.retrieved_chunk_ids}")
    print(f"A: {result.answer}")
    print("Trace:")
    for step in result.tool_call_trace:
        print(f"  {step}")
    print()


--- keyword ---
Q: What does PCEval stand for?
Retrieved: ['arxiv:2601.02404#0', 'arxiv:2601.02404#1', 'arxiv:2601.02404#2']
A: PCEVAL stands for Physical Computing Evaluation. It is a benchmark designed for evaluating the physical computing capabilities of large language models (LLMs), assessing their ability to generate circuits and produce compatible code for physical hardware projects automatically and without human assessment [arxiv:2601.02404#0].
Trace:
  {'thought': 'I need to find the full form or definition of PCEval, so I will use keyword search to find exact mentions of PCEval in the corpus.', 'action': 'search_bm25', 'action_input': 'PCEval', 'observation': "Retrieved: ['arxiv:2601.02404#0', 'arxiv:2601.02404#1', 'arxiv:2601.02404#2', 'arxiv:2601.11580#0', 'arxiv:2601.11580#1']"}
  {'thought': 'I have documents mentioning PCEval but need to extract the exact full form or definition, so I will perform a dense search with a query focused on defining PCEval to find relevant ex

--- paraphrase ---
Q: Why do repeated image generations from the same text prompt in diffusion models end up looking so similar to each other?
Retrieved: ['arxiv:2601.00090#0', 'arxiv:2601.00090#1', 'arxiv:2601.00090#2']
A: Repeated image generations from the same text prompt in diffusion models end up looking very similar due to a phenomenon called mode collapse. This occurs because the models tend to produce nearly identical outputs across different random seeds when given the same prompt, limiting the diversity of generated images. The initial noise inputs play a crucial role in this, but simply relying on random seeds often fails to yield varied results. Thus, the model repeatedly generates images that are visually alike rather than diverse [arxiv:2601.00090#0, #1].
Trace:
  {'thought': "I need to understand the mechanism of diffusion models and how randomness or noise is introduced during image generation to explain why repeated generations from the same prompt look similar. I'll 

--- multi_hop ---
Q: The two photonics-AI-systems papers in this corpus each address a different part of the same challenge. What does each one focus on?
Retrieved: ['arxiv:2601.00129#0', 'arxiv:2601.00130#0', 'arxiv:2601.00129#2']
A: The two photonics-AI-systems papers focus on complementary aspects of the challenge of realizing large-scale photonics-empowered AI systems:

1. The first paper, "Toward Large-Scale Photonics-Empowered AI Systems: From Physical Design Automation to System-Algorithm Co-Exploration" [arxiv:2601.00129], emphasizes the need for **physical design automation (EPDA)** and **system-algorithm co-exploration**. It addresses the integration challenges across devices, circuits, architectures, interconnect fabrics, and learning algorithms, highlighting the importance of scalable, manufacturable photonic integrated circuits and the incorporation of physical non-idealities and packaging/interface costs into system and algorithm design. This paper presents a cross-layer 

--- filter ---
Q: Among the cs.LG papers in this corpus, which one addresses diagnosing a pregnancy complication using deep learning?
Retrieved: ['arxiv:2601.00907#0', 'arxiv:2601.00907#2', 'arxiv:2601.00907#1']
A: The paper titled "Placenta Accreta Spectrum Detection using Multimodal Deep Learning" addresses diagnosing a pregnancy complication (Placenta Accreta Spectrum, PAS) using deep learning. It develops a multimodal deep learning framework that integrates 3D MRI and 2D Ultrasound scans to improve early and accurate prenatal diagnosis of PAS, a life-threatening obstetric condition. The model uses a 3D DenseNet121-Vision Transformer for MRI and a 2D ResNet50 for Ultrasound, achieving superior diagnostic performance compared to unimodal models [arxiv:2601.00907#0, #2].
Trace:
  {'thought': "I need to find papers in cs.LG that discuss diagnosing pregnancy complications using deep learning. Using keyword search with terms like 'pregnancy complication' and 'deep learning' should help l

## 7. Where this pattern FAILS

**PENDING: real findings from a one-off real-embeddings run.** A real `OPENAI_API_KEY` was not yet
available in the environment when this notebook was authored. This section will be replaced with a
static table of genuine hit@k/mrr failures (matching the format used in `02_bm25.ipynb`/
`04_rerank.ipynb` section 7), computed via a real, uncommitted exploratory run once a key is
available. The claim will be labeled with the date it was run
and its actual dollar cost, and will not be presented as live-executed cell output, to avoid
implying a mock re-run reproduces it (see this notebook's top-of-file disclaimer).


## 8. Copy-paste snippet

Meant for pasting into your own project, not executed as a cell in this notebook.

```python
"""Minimal agentic retrieval + generation, no eval harness."""
from recipes.embeddings import get_embedder
from recipes.agentic import make_retrieve_and_answer
from recipes.llm import get_llm

corpus_by_id = {}  # {chunk_id: {"text": ..., ...}, ...} -- fill in your own chunks
embedder = get_embedder()
llm = get_llm()

retrieve_and_answer = make_retrieve_and_answer(corpus_by_id, embedder=embedder, llm=llm)
result = retrieve_and_answer("your question here", k=5)
print(result.answer)
print(result.tool_call_trace)
```
